# BWGNN v2 — mejorarlo y portarlo a Yelp-NYC (Google Colab, T4)

Proyecto **fake-review-detector / CheckGraph**. Segunda vuelta sobre grafo tras
`colab/bwgnn_yelpchi.ipynb`, que ya midió que **BWGNN hetero** (relaciones separadas,
nunca fusionadas) da el mejor resultado de grafo del proyecto sobre YelpChi:

| Modelo | AUC | AP |
|---|---|---|
| MLP sin grafo | 0,8448 | 0,5387 |
| GCN homo | 0,8041 | 0,4611 |
| BWGNN homo | 0,8472 | 0,5726 |
| **BWGNN hetero** | **0,9120** | **0,6927** |

Este notebook tiene dos partes:

- **Parte A — mejorar BWGNN hetero sobre YelpChi.** La pista de la sesión anterior:
  `best_epoch` fue 196 de 200 — el modelo seguía mejorando cuando se acabó el
  entrenamiento. Se prueba más entrenamiento con early stopping real, una búsqueda
  de hiperparámetros optimizando **AP en validación** (nunca en test), y un
  ensemble de semillas.
- **Parte B — portar BWGNN hetero a Yelp-NYC** (359.052 reviews, el dataset real
  del producto), usando las 24 features label-free ya construidas en
  `features_behavior.py` y empaquetadas en `data_bundles/yelpnyc_bundle.npz`.
  Hasta ahora esto era inviable por memoria (~73M aristas en una de las
  relaciones) — se resuelve con un truco exacto explicado en la Parte B.

Todo el modelo es **label-free**: ninguna feature usa `is_fake` para calcularse.
El clasificador se entrena de forma supervisada, pero el cliente no necesita
aportar etiquetas para que las features existan.

**Tiempo estimado en una T4 gratuita**: Parte A ~20-35 min, Parte B ~10-30 min
(estimación por extrapolación de los tiempos ya medidos en `bwgnn_yelpchi.ipynb`,
no cronometrado en una sesión real — si algo tarda mucho más de lo aquí anotado,
puede ser señal de que algo va mal, no solo lentitud esperable).


## 0. Comprobación del entorno

Si esto dice que **no hay GPU**: `Entorno de ejecución` → `Cambiar tipo de entorno
de ejecución` → `T4 GPU`. El notebook funciona en CPU pero mucho más lento.

In [ ]:
import platform
import torch
import numpy as np
import scipy
import sklearn

print("python      ", platform.python_version())
print("torch       ", torch.__version__)
print("numpy       ", np.__version__)
print("scipy       ", scipy.__version__)
print("scikit-learn", sklearn.__version__)
print()
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("SIN GPU -- cambia el entorno de ejecucion antes de seguir (ver arriba).")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


---
# Parte A — mejorar BWGNN hetero sobre YelpChi

## A.1 Descarga del dataset

Mismo fichero que usa `data.py::load_yelpchi_graph_dataset()` en el repo y que ya
usó `bwgnn_yelpchi.ipynb` — los sanity-checks de abajo confirman que es exactamente
el mismo dato (45.954 nodos, 6.677 positivos).

In [ ]:
import time
import urllib.request
import zipfile
from pathlib import Path

YELPCHI_URL = "https://github.com/YingtongDou/CARE-GNN/raw/master/data/YelpChi.zip"
DATA_DIR = Path("/content/data") if Path("/content").exists() else Path("./data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_DIR / "YelpChi.zip"
MAT_PATH = DATA_DIR / "YelpChi.mat"

if not MAT_PATH.exists():
    if not ZIP_PATH.exists():
        print("Descargando", YELPCHI_URL)
        t0 = time.time()
        urllib.request.urlretrieve(YELPCHI_URL, ZIP_PATH)
        print(f"  {ZIP_PATH.stat().st_size / 1024**2:.1f} MB en {time.time() - t0:.1f}s")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        names = zf.namelist()
        assert "YelpChi.mat" in names, f"No se encuentra YelpChi.mat en el zip: {names}"
        zf.extract("YelpChi.mat", DATA_DIR)

print(f"{MAT_PATH} -> {MAT_PATH.stat().st_size / 1024**2:.1f} MB")


In [ ]:
import scipy.io
import scipy.sparse as sp

t0 = time.time()
mat = scipy.io.loadmat(MAT_PATH)
print(f"loadmat: {time.time() - t0:.1f}s")

yc_features = np.asarray(mat["features"].todense(), dtype=np.float32)
yc_labels = np.asarray(mat["label"]).reshape(-1).astype(np.int64)
YC_RELATIONS = {
    "homo": mat["homo"],
    "net_rur": mat["net_rur"],
    "net_rtr": mat["net_rtr"],
    "net_rsr": mat["net_rsr"],
}

YC_N_NODES, YC_N_FEATS = yc_features.shape

assert yc_features.shape == (45954, 32), yc_features.shape
assert yc_labels.shape == (45954,), yc_labels.shape
assert int(yc_labels.sum()) == 6677, int(yc_labels.sum())
assert 0.0 <= yc_features.min() and yc_features.max() <= 1.0 + 1e-6

print(f"nodos {YC_N_NODES:,}  features {YC_N_FEATS}  positivos {int(yc_labels.sum()):,} "
      f"({yc_labels.mean():.2%})")


## A.2 Laplaciano unificado: sparse (YelpChi) y por scatter (Yelp-NYC)

Dos formas de calcular $L = I - D^{-1/2} A D^{-1/2}$ (convención con self-loops:
$\hat A = A + I$, $\hat D$ = grado de $\hat A$ — igual que `bwgnn_yelpchi.ipynb`):

- **`SparseLap`**: construye la matriz sparse explícita y multiplica. Válido para
  cualquier grafo, usado aquí para YelpChi (45.954 nodos, factible).
- **`GroupLap`** (Parte B): cuando la relación es una **unión de cliques
  disjuntos** (verificado en sesiones anteriores del proyecto para
  `reviewer_id`/`business_id+rating`/etc.), todos los nodos de un grupo tienen el
  mismo grado y $D^{-1/2} A D^{-1/2}$ colapsa a una media por grupo — calculable
  con `index_add_` (scatter), sin construir ninguna arista. Verificado por
  separado que ambos métodos dan resultados numéricamente idénticos (diferencia
  <1e-6 en float32) antes de escribir este notebook.

Las dos exponen la misma interfaz `.mm(h)`, así que el resto del código (bancos de
wavelets, `GraphNet`) es idéntico en las dos partes.

In [ ]:
def _build_sparse_laplacian(adj, device):
    # L = I - D^-1/2 A D^-1/2, con self-loops (A_hat = A + I). A se simetriza y
    # binariza antes de normalizar -- misma convencion que bwgnn_yelpchi.ipynb.
    n = adj.shape[0]
    a = adj.tocoo()
    diag = np.arange(n, dtype=np.int64)
    rows = np.concatenate([a.row.astype(np.int64), a.col.astype(np.int64), diag])
    cols = np.concatenate([a.col.astype(np.int64), a.row.astype(np.int64), diag])
    a = sp.coo_matrix((np.ones(rows.size, dtype=np.float32), (rows, cols)), shape=(n, n)).tocsr()
    a.data[:] = 1.0
    a = a.tocoo()

    deg = np.asarray(a.sum(axis=1)).reshape(-1)
    d_inv_sqrt = 1.0 / np.sqrt(np.maximum(deg, 1.0))
    off = -(d_inv_sqrt[a.row] * d_inv_sqrt[a.col]).astype(np.float32)

    idx = np.vstack([
        np.concatenate([a.row.astype(np.int64), diag]),
        np.concatenate([a.col.astype(np.int64), diag]),
    ])
    val = np.concatenate([off, np.ones(n, dtype=np.float32)])
    lap = torch.sparse_coo_tensor(torch.from_numpy(idx), torch.from_numpy(val), (n, n))
    return lap.coalesce().to(device)


class SparseLap:
    """L @ h via matriz sparse explicita. Para grafos donde construir las
    aristas es factible (YelpChi, 45.954 nodos)."""

    def __init__(self, adj, device):
        self.L = _build_sparse_laplacian(adj, device)

    def mm(self, h):
        return torch.sparse.mm(self.L, h)


class GroupLap:
    """L @ h via scatter sobre ids de grupo, SIN construir ninguna arista.
    Solo exacto si la relacion es una union de cliques disjuntos (cada nodo
    conectado a TODOS los demas de su grupo y a nadie mas) -- verificado para
    las relaciones de este proyecto en sesiones anteriores (Louvain devuelve
    exactamente el mismo numero de comunidades que el agrupado por columnas).
    """

    def __init__(self, group_ids, device):
        group_ids = torch.as_tensor(np.asarray(group_ids), dtype=torch.long, device=device)
        self.group_ids = group_ids
        self.n_groups = int(group_ids.max().item()) + 1
        ones = torch.ones_like(group_ids, dtype=torch.float32)
        size = torch.zeros(self.n_groups, device=device).index_add_(0, group_ids, ones)
        self.size_per_node = size[group_ids]

    def mm(self, h):
        # Con self-loops, un nodo de un clique de tamano m tiene grado m (m-1
        # vecinos + 1 self), y D^-1/2 A D^-1/2 sobre ese clique colapsa a A/m:
        # L @ h = h - media_del_grupo_INCLUYENDO al propio nodo (no LOO).
        group_sum = torch.zeros(self.n_groups, h.shape[1], device=h.device, dtype=h.dtype)
        group_sum.index_add_(0, self.group_ids, h)
        mean_incl_self = group_sum[self.group_ids] / self.size_per_node.unsqueeze(1)
        return h - mean_incl_self


YC_LAPS = {}
for key, adj in YC_RELATIONS.items():
    t0 = time.time()
    YC_LAPS[key] = SparseLap(adj, DEVICE)
    print(f"{key:<9} ({time.time() - t0:.1f}s)")

# Sanity check: los autovalores de L normalizado estan en [0, 2], asi que
# ||L x|| no puede superar 2*||x||. Si falla, la normalizacion esta mal.
_x = torch.randn(YC_N_NODES, 8, device=DEVICE)
for key, lap in YC_LAPS.items():
    ratio = (lap.mm(_x).norm() / _x.norm()).item()
    assert ratio <= 2.0 + 1e-3, (key, ratio)
    print(f"  {key:<9} ||L x|| / ||x|| = {ratio:.3f}  (debe ser <= 2)")
del _x


## A.3 Bancos de filtros y modelos

Idénticos a `bwgnn_yelpchi.ipynb`, salvo que ahora llaman a `lap.mm(h)` en vez de
`torch.sparse.mm(lap, h)` — así el mismo código sirve para `SparseLap` (Parte A)
y `GroupLap` (Parte B) sin duplicar la arquitectura.

In [ ]:
import scipy.special
import torch.nn as nn
import torch.nn.functional as F


def beta_wavelet_thetas(d):
    thetas = []
    for p in range(d + 1):
        norm = scipy.special.beta(p + 1, d + 1 - p)
        coeffs = [0.0] * (d + 1)
        for k in range(p, d + 1):
            coeffs[k] = float(
                scipy.special.comb(d - p, k - p, exact=True) * ((-1) ** (k - p)) / (2 ** k) / norm
            )
        thetas.append(coeffs)
    return thetas


assert np.allclose(beta_wavelet_thetas(2), [[3.0, -3.0, 0.75], [0.0, 3.0, -1.5], [0.0, 0.0, 0.75]])


class BetaWaveletBank(nn.Module):
    def __init__(self, d=2):
        super().__init__()
        self.d = d
        self.register_buffer("thetas", torch.tensor(beta_wavelet_thetas(d), dtype=torch.float32))

    @property
    def n_out(self):
        return self.d + 1

    def forward(self, lap, h):
        powers = [h]
        for _ in range(self.d):
            powers.append(lap.mm(powers[-1]))
        out = []
        for i in range(self.d + 1):
            acc = self.thetas[i, 0] * powers[0]
            for k in range(1, self.d + 1):
                acc = acc + self.thetas[i, k] * powers[k]
            out.append(acc)
        return out


class LowPassBank(nn.Module):
    def __init__(self, d=2):
        super().__init__()
        self.d = d

    @property
    def n_out(self):
        return 1

    def forward(self, lap, h):
        for _ in range(self.d):
            h = h - 0.5 * lap.mm(h)
        return [h]


class GraphNet(nn.Module):
    def __init__(self, in_feats, h_feats, n_rel, bank_cls=BetaWaveletBank, d=2, dropout=0.1):
        super().__init__()
        self.banks = nn.ModuleList([bank_cls(d) for _ in range(n_rel)])
        n_cat = h_feats * sum(b.n_out for b in self.banks)
        self.lin1 = nn.Linear(in_feats, h_feats)
        self.lin2 = nn.Linear(h_feats, h_feats)
        self.lin3 = nn.Linear(n_cat, h_feats)
        self.lin4 = nn.Linear(h_feats, 2)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, laps):
        h = self.drop(F.relu(self.lin1(x)))
        h = F.relu(self.lin2(h))
        outs = []
        for bank, lap in zip(self.banks, laps):
            outs.extend(bank(lap, h))
        h = torch.cat(outs, dim=-1)
        h = self.drop(F.relu(self.lin3(h)))
        return self.lin4(h)


class MLPNet(nn.Module):
    def __init__(self, in_feats, h_feats, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_feats, h_feats), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h_feats, h_feats), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h_feats, 2),
        )

    def forward(self, x, laps):
        return self.net(x)


## A.4 Protocolo de evaluación

Igual que `bwgnn_yelpchi.ipynb`: split train/val/test estratificado, época elegida
por **AP en validación**, evaluación final en test. Se añade `patience` para
early stopping real (parar cuando no mejora, no solo entrenar un nº fijo de
épocas) — necesario para poder subir el techo de épocas sin disparar el tiempo.

**Umbral de ruido**: con el test de YelpChi (~9.191 filas a 40% train, ~13.786 a
1%), diferencias menores de **0,01 de AUC o 0,02 de AP** entre variantes son
ruido de muestreo, no una mejora real — no perseguir décimas dentro de ese
margen.

In [ ]:
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42


def topk_metrics(y_true, scores, frac):
    k = max(1, int(round(len(y_true) * frac)))
    top = np.argsort(-scores)[:k]
    hits = int(y_true[top].sum())
    base = float(y_true.mean())
    precision = hits / k
    return {
        "k": k,
        "precision": precision,
        "recall": hits / max(1, int(y_true.sum())),
        "lift": (precision / base) if base > 0 else float("nan"),
    }


def evaluate(y_true, scores):
    out = {
        "n": int(len(y_true)),
        "n_pos": int(y_true.sum()),
        "base_rate": float(y_true.mean()),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
    }
    for frac in (0.01, 0.05, 0.10):
        out[f"top_{int(frac * 100)}pct"] = topk_metrics(y_true, scores, frac)
    return out


def split_masks(y, train_frac, val_frac, seed=SEED):
    idx = np.arange(len(y))
    tr, rest = train_test_split(idx, train_size=train_frac, stratify=y, random_state=seed)
    va, te = train_test_split(
        rest, train_size=val_frac / (1.0 - train_frac), stratify=y[rest], random_state=seed
    )
    return tr, va, te


YC_REGIMES = {"40pct": (0.40, 0.20), "1pct": (0.01, 0.01)}
for name, (trf, vaf) in YC_REGIMES.items():
    tr, va, te = split_masks(yc_labels, trf, vaf)
    assert len(set(tr) & set(te)) == 0 and len(set(va) & set(te)) == 0
    print(f"{name:<6} train {len(tr):>6,} ({yc_labels[tr].mean():.3%} pos)  "
          f"val {len(va):>6,}  test {len(te):>6,} ({yc_labels[te].mean():.3%} pos)")


## A.5 Función de entrenamiento con early stopping real

Extiende la del notebook anterior con `patience`: si `patience` es `None`,
entrena las `epochs` fijas exactas (para reproducir el baseline tal cual);
si tiene un valor, para en cuanto el AP de validación no mejora durante esas
épocas seguidas.

In [ ]:
def train_one(features, labels, laps_dict, model_name, model_cfg, regime, regimes,
              epochs=200, hidden=64, d=2, lr=0.01, weight_decay=0.0, dropout=0.1,
              patience=None, seed=SEED, verbose=True, device=None):
    device = device or DEVICE
    n_feats = features.shape[1]
    train_frac, val_frac = regimes[regime]
    tr, va, te = split_masks(labels, train_frac, val_frac, seed=seed)

    torch.manual_seed(seed)
    np.random.seed(seed)

    laps = [laps_dict[r] for r in model_cfg["rels"]]
    if model_cfg["kind"] == "mlp":
        model = MLPNet(n_feats, hidden, dropout=dropout).to(device)
    else:
        model = GraphNet(n_feats, hidden, len(laps), bank_cls=model_cfg["bank"],
                          d=d, dropout=dropout).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_pos = int(labels[tr].sum())
    pos_weight = (len(tr) - n_pos) / max(1, n_pos)
    class_w = torch.tensor([1.0, pos_weight], dtype=torch.float32, device=device)

    X_t = torch.from_numpy(features).to(device)
    Y_t = torch.from_numpy(labels).to(device)
    tr_t = torch.from_numpy(tr).to(device)

    best_ap, best_scores, best_epoch, no_improve = -1.0, None, -1, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        loss = F.cross_entropy(model(X_t, laps)[tr_t], Y_t[tr_t], weight=class_w)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            scores = F.softmax(model(X_t, laps), dim=1)[:, 1].float().cpu().numpy()
        ap_val = average_precision_score(labels[va], scores[va])
        if ap_val > best_ap:
            best_ap, best_scores, best_epoch, no_improve = ap_val, scores.copy(), ep, 0
        else:
            no_improve += 1

        if verbose and (ep == 1 or ep % 100 == 0):
            print(f"    ep {ep:>4}  loss {loss.item():.4f}  val_AP {ap_val:.4f}  "
                  f"(mejor {best_ap:.4f} en ep {best_epoch})")

        if patience is not None and no_improve >= patience:
            if verbose:
                print(f"    early stop en ep {ep} (sin mejora desde ep {best_epoch})")
            break

    best_thr, best_f1_val = 0.5, -1.0
    for thr in np.unique(np.quantile(best_scores[va], np.linspace(0.01, 0.99, 99))):
        f1_val = f1_score(labels[va], (best_scores[va] >= thr).astype(int),
                           average="macro", zero_division=0)
        if f1_val > best_f1_val:
            best_f1_val, best_thr = f1_val, float(thr)

    res = evaluate(labels[te], best_scores[te])
    res.update(
        model=model_name, regime=regime, best_epoch=best_epoch,
        val_average_precision=float(best_ap), train_seconds=round(time.time() - t0, 1),
        epochs_ran=ep, threshold_from_val=best_thr,
        f1_macro_val_thr=float(f1_score(labels[te], (best_scores[te] >= best_thr).astype(int),
                                         average="macro", zero_division=0)),
        n_params=int(sum(p.numel() for p in model.parameters())),
        hidden=hidden, wavelet_order=d, lr=lr, weight_decay=weight_decay, dropout=dropout,
        patience=patience, device=str(device),
    )
    return res, tr, va, te


BWGNN_HETERO_CFG = dict(kind="graph", rels=["net_rur", "net_rtr", "net_rsr"], bank=BetaWaveletBank)
MLP_CFG = dict(kind="mlp", rels=[])


## A.6 Paso 0 — reproducir el baseline (ancla de cordura)

Antes de mejorar nada, confirmar que este código reproduce lo que ya se midió
(0,9120 AUC / 0,6927 AP a 40% train, `epochs=200` fijas, sin early stopping). Si
esto no coincide de cerca, el problema está en el refactor a `SparseLap`/`GroupLap`,
no en las mejoras — parar aquí y diagnosticar antes de seguir.

In [ ]:
REF_BASELINE_40PCT = {"roc_auc": 0.9120, "average_precision": 0.6927}

baseline_res, *_ = train_one(
    yc_features, yc_labels, YC_LAPS, "bwgnn_hetero", BWGNN_HETERO_CFG, "40pct", YC_REGIMES,
    epochs=200, hidden=64, d=2, lr=0.01, weight_decay=0.0, patience=None, verbose=True,
)
print(f"\nReproducido:  AUC {baseline_res['roc_auc']:.4f}  AP {baseline_res['average_precision']:.4f}")
print(f"Referencia:   AUC {REF_BASELINE_40PCT['roc_auc']:.4f}  AP {REF_BASELINE_40PCT['average_precision']:.4f}")

d_auc = abs(baseline_res["roc_auc"] - REF_BASELINE_40PCT["roc_auc"])
d_ap = abs(baseline_res["average_precision"] - REF_BASELINE_40PCT["average_precision"])
print(f"Diferencia:   AUC {d_auc:.4f}  AP {d_ap:.4f}")
if d_auc > 0.03 or d_ap > 0.03:
    print("\nAVISO: la diferencia es mayor de lo esperable por aleatoriedad de "
          "inicializacion (>0.03). Revisar el refactor de SparseLap antes de seguir "
          "-- las mejoras de abajo no seran comparables si este numero no cuadra.")
else:
    print("\nOK: dentro del margen esperable por semillas/no-determinismo de GPU. Seguimos.")


## A.7 Mejora 1 — más épocas con early stopping real

`best_epoch` fue 196 de 200 en la sesión anterior: el modelo seguía mejorando
cuando se acabó el entrenamiento. Se sube el techo a 1500 con `patience=150`
(para de verdad cuando deja de mejorar, no antes).

In [ ]:
mejora1_res, *_ = train_one(
    yc_features, yc_labels, YC_LAPS, "bwgnn_hetero", BWGNN_HETERO_CFG, "40pct", YC_REGIMES,
    epochs=1500, hidden=64, d=2, lr=0.01, weight_decay=0.0, patience=150, verbose=True,
)
print(f"\nBaseline (200 ep fijas):  AUC {baseline_res['roc_auc']:.4f}  AP {baseline_res['average_precision']:.4f}")
print(f"+ mas epocas ({mejora1_res['epochs_ran']} ep, mejor en {mejora1_res['best_epoch']}):  "
      f"AUC {mejora1_res['roc_auc']:.4f}  AP {mejora1_res['average_precision']:.4f}")
print(f"Delta AP: {mejora1_res['average_precision'] - baseline_res['average_precision']:+.4f}")


## A.8 Mejora 2 — búsqueda de `wavelet_order` (d) y `hidden`

Rejilla pequeña a propósito (mantener el notebook en un tiempo razonable):
`d ∈ {2, 3, 4}` × `hidden ∈ {64, 128}` = 6 combinaciones, elegidas por **AP en
validación**, nunca en test. `epochs=600, patience=80` en esta fase de búsqueda
(más corto que la definitiva, para no gastar todo el tiempo aquí).

In [ ]:
grid_a_results = []
for d_try in (2, 3, 4):
    for hidden_try in (64, 128):
        r, *_ = train_one(
            yc_features, yc_labels, YC_LAPS, "bwgnn_hetero", BWGNN_HETERO_CFG, "40pct", YC_REGIMES,
            epochs=600, hidden=hidden_try, d=d_try, lr=0.01, weight_decay=0.0,
            patience=80, verbose=False,
        )
        grid_a_results.append(r)
        print(f"d={d_try} hidden={hidden_try:<4} -> val_AP {r['val_average_precision']:.4f}  "
              f"test AUC {r['roc_auc']:.4f} AP {r['average_precision']:.4f}  "
              f"({r['epochs_ran']} ep, {r['train_seconds']}s)")

best_grid_a = max(grid_a_results, key=lambda r: r["val_average_precision"])
BEST_D, BEST_HIDDEN = best_grid_a["wavelet_order"], best_grid_a["hidden"]
print(f"\nMejor por val_AP: d={BEST_D} hidden={BEST_HIDDEN}  "
      f"(test AUC {best_grid_a['roc_auc']:.4f}  AP {best_grid_a['average_precision']:.4f})")


## A.9 Mejora 3 — refinar `lr` y `weight_decay` sobre el mejor `d`/`hidden`

Con `d`/`hidden` ya fijados, una rejilla pequeña de regularización:
`weight_decay ∈ {0, 1e-4}` × `lr ∈ {0,01, 0,005}` = 4 combinaciones.

In [ ]:
grid_b_results = []
for wd_try in (0.0, 1e-4):
    for lr_try in (0.01, 0.005):
        r, *_ = train_one(
            yc_features, yc_labels, YC_LAPS, "bwgnn_hetero", BWGNN_HETERO_CFG, "40pct", YC_REGIMES,
            epochs=600, hidden=BEST_HIDDEN, d=BEST_D, lr=lr_try, weight_decay=wd_try,
            patience=80, verbose=False,
        )
        grid_b_results.append(r)
        print(f"lr={lr_try} wd={wd_try:<7} -> val_AP {r['val_average_precision']:.4f}  "
              f"test AUC {r['roc_auc']:.4f} AP {r['average_precision']:.4f}  "
              f"({r['epochs_ran']} ep, {r['train_seconds']}s)")

best_grid_b = max(grid_b_results, key=lambda r: r["val_average_precision"])
BEST_LR, BEST_WD = best_grid_b["lr"], best_grid_b["weight_decay"]
print(f"\nMejor por val_AP: lr={BEST_LR} weight_decay={BEST_WD}")

BEST_CONFIG = dict(hidden=BEST_HIDDEN, d=BEST_D, lr=BEST_LR, weight_decay=BEST_WD, dropout=0.1)
print(f"\nBEST_CONFIG (se reusa en la Parte B): {BEST_CONFIG}")


## A.10 Mejora 4 — ensemble de semillas

Con `BEST_CONFIG` ya fijado, 5 semillas independientes, promediadas por rango
(más robusto que promediar probabilidades crudas cuando las escalas no son
directamente comparables entre corridas).

In [ ]:
N_SEEDS = 5
seed_scores = []
seed_res = []
for seed_try in range(N_SEEDS):
    r, tr_s, va_s, te_s = train_one(
        yc_features, yc_labels, YC_LAPS, "bwgnn_hetero", BWGNN_HETERO_CFG, "40pct", YC_REGIMES,
        epochs=1500, patience=150, seed=seed_try, verbose=False, **BEST_CONFIG,
    )
    seed_res.append(r)
    print(f"seed {seed_try}: val_AP {r['val_average_precision']:.4f}  "
          f"test AUC {r['roc_auc']:.4f} AP {r['average_precision']:.4f}")

# Todas las semillas usan el MISMO split (seed_try solo cambia la inicializacion
# del modelo, no `split_masks` -- ojo, train_one usa `seed` tambien para el
# split; para el ensemble de verdad interesa variar solo la inicializacion del
# modelo con un split fijo. Se corrige explicitamente:
tr_fix, va_fix, te_fix = split_masks(yc_labels, *YC_REGIMES["40pct"], seed=SEED)
ensemble_ranks = None
for seed_try in range(N_SEEDS):
    torch.manual_seed(1000 + seed_try)
    np.random.seed(1000 + seed_try)
    model = GraphNet(YC_N_FEATS, BEST_CONFIG["hidden"], len(BWGNN_HETERO_CFG["rels"]),
                      bank_cls=BWGNN_HETERO_CFG["bank"], d=BEST_CONFIG["d"],
                      dropout=BEST_CONFIG["dropout"]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=BEST_CONFIG["lr"],
                            weight_decay=BEST_CONFIG["weight_decay"])
    n_pos = int(yc_labels[tr_fix].sum())
    pos_weight = (len(tr_fix) - n_pos) / max(1, n_pos)
    class_w = torch.tensor([1.0, pos_weight], dtype=torch.float32, device=DEVICE)
    X_t = torch.from_numpy(yc_features).to(DEVICE)
    Y_t = torch.from_numpy(yc_labels).to(DEVICE)
    tr_t = torch.from_numpy(tr_fix).to(DEVICE)
    laps = [YC_LAPS[r] for r in BWGNN_HETERO_CFG["rels"]]

    best_ap_s, best_scores_s, no_improve = -1.0, None, 0
    for ep in range(1, 1501):
        model.train()
        opt.zero_grad()
        loss = F.cross_entropy(model(X_t, laps)[tr_t], Y_t[tr_t], weight=class_w)
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            scores = F.softmax(model(X_t, laps), dim=1)[:, 1].float().cpu().numpy()
        ap_v = average_precision_score(yc_labels[va_fix], scores[va_fix])
        if ap_v > best_ap_s:
            best_ap_s, best_scores_s, no_improve = ap_v, scores.copy(), 0
        else:
            no_improve += 1
            if no_improve >= 150:
                break

    ranks = np.argsort(np.argsort(best_scores_s))  # rango creciente, mismo tamano para todos
    ensemble_ranks = ranks.astype(np.float64) if ensemble_ranks is None else ensemble_ranks + ranks

ensemble_scores = ensemble_ranks / N_SEEDS
ensemble_res = evaluate(yc_labels[te_fix], ensemble_scores[te_fix])
print(f"\nEnsemble de {N_SEEDS} semillas (mismo split, distinta init): "
      f"AUC {ensemble_res['roc_auc']:.4f}  AP {ensemble_res['average_precision']:.4f}")


## A.11 Resumen Parte A

In [ ]:
import pandas as pd

resumen_a = pd.DataFrame([
    {"variante": "baseline (200 ep fijas)", "AUC": baseline_res["roc_auc"], "AP": baseline_res["average_precision"]},
    {"variante": "+ mas epocas + early stop", "AUC": mejora1_res["roc_auc"], "AP": mejora1_res["average_precision"]},
    {"variante": f"+ d={BEST_D} hidden={BEST_HIDDEN}", "AUC": best_grid_a["roc_auc"], "AP": best_grid_a["average_precision"]},
    {"variante": f"+ lr={BEST_LR} wd={BEST_WD}", "AUC": best_grid_b["roc_auc"], "AP": best_grid_b["average_precision"]},
    {"variante": f"+ ensemble {N_SEEDS} semillas", "AUC": ensemble_res["roc_auc"], "AP": ensemble_res["average_precision"]},
])
resumen_a["delta_AP_vs_baseline"] = resumen_a["AP"] - baseline_res["average_precision"]
print("Umbral de ruido: diferencias < 0.01 AUC / 0.02 AP no son mejoras reales, son varianza de muestreo.")
resumen_a


---
# Parte B — BWGNN hetero sobre Yelp-NYC

Yelp-NYC (359.052 reviews, 160.225 reviewers, 923 negocios, 10,27% de fraude) es
el dataset real del producto, a diferencia de YelpChi (banco de pruebas). Hasta
ahora no se podía portar BWGNN aquí por dos motivos, los dos resueltos:

1. **BWGNN necesita features de nodo y Yelp-NYC no las tenía** — ahora sí: las 24
   features label-free de `features_behavior.py`, empaquetadas en
   `data_bundles/yelpnyc_bundle.npz`.
2. **Escala**: una de las relaciones (`business_id, rating`) tiene decenas de
   millones de aristas si se construyen explícitamente — inviable en una T4.
   Se resuelve con `GroupLap` (ya definido en A.2): como la relación es una
   unión de cliques disjuntos, la propagación se calcula con `index_add_`
   (scatter) sin construir ni una sola arista.

## B.1 Descargar el bundle de features

Si el notebook corre desde GitHub (pestaña "GitHub" al abrir en Colab), se
descarga directamente del repo. Si falla (por ejemplo, cambios sin pushear
todavía), se puede subir el fichero a mano al panel de la izquierda.

In [ ]:
YELPNYC_BUNDLE_URL = (
    "https://raw.githubusercontent.com/PabloMarpar/fake-review-detector/main/"
    "data_bundles/yelpnyc_bundle.npz"
)
BUNDLE_PATH = DATA_DIR / "yelpnyc_bundle.npz"

if not BUNDLE_PATH.exists():
    try:
        print("Descargando", YELPNYC_BUNDLE_URL)
        urllib.request.urlretrieve(YELPNYC_BUNDLE_URL, BUNDLE_PATH)
        print(f"  {BUNDLE_PATH.stat().st_size / 1024**2:.1f} MB")
    except Exception as e:
        print(f"No se pudo descargar automaticamente ({e}).")
        print("Sube 'yelpnyc_bundle.npz' a mano al panel de archivos de la izquierda,")
        print(f"y colocalo en: {BUNDLE_PATH}")

assert BUNDLE_PATH.exists(), f"Falta {BUNDLE_PATH} -- sin este fichero no se puede continuar la Parte B."

bundle = np.load(BUNDLE_PATH, allow_pickle=False)
ny_X_raw = bundle["X"].astype(np.float32)
ny_feature_names = [str(s) for s in bundle["feature_names"]]
ny_y = bundle["y"].astype(np.int64)
ny_reviewer_id = bundle["reviewer_id"].astype(np.int64)
ny_business_id = bundle["business_id"].astype(np.int64)
ny_rating = bundle["rating"].astype(np.int64)
ny_date_days = bundle["date_days"].astype(np.int64)

print(f"X: {ny_X_raw.shape}  features: {ny_feature_names}")
print(f"y: {ny_y.sum()} fraude / {len(ny_y)} ({ny_y.mean():.4f})")
print(f"reviewers unicos: {ny_reviewer_id.max() + 1}, negocios unicos: {ny_business_id.max() + 1}")
assert not np.isnan(ny_X_raw).any(), "hay NaN en el bundle -- revisar features_behavior.py"


## B.2 Relaciones como `GroupLap` — sin construir ninguna arista

Tres relaciones, todas label-free:
- `rev` = `reviewer_id` (grupos densos donde el reviewer repite)
- `biz_rating` = `(business_id, rating)` (grupos grandes, cientos-miles por grupo)
- `biz_rating_week` = `(business_id, rating, semana)` (mayoría singletons — se
  mide si aporta o solo añade ruido)

In [ ]:
ny_week = ny_date_days // 7

_, ny_group_rev = np.unique(ny_reviewer_id, return_inverse=True)
_, ny_group_biz_rating = np.unique(ny_business_id * 10 + ny_rating, return_inverse=True)
_key_brw = (ny_business_id.astype(np.int64) * 10 + ny_rating) * 10000 + ny_week
_, ny_group_biz_rating_week = np.unique(_key_brw, return_inverse=True)

NY_GROUPS = {
    "rev": ny_group_rev,
    "biz_rating": ny_group_biz_rating,
    "biz_rating_week": ny_group_biz_rating_week,
}
for name, g in NY_GROUPS.items():
    sizes = np.bincount(g)
    print(f"{name:<16} {len(sizes):>7,} grupos  mediana={np.median(sizes):.0f}  "
          f"max={sizes.max():,}  singletons={int((sizes == 1).sum()):,} "
          f"({(sizes == 1).mean():.1%})")

NY_LAPS = {name: GroupLap(g, DEVICE) for name, g in NY_GROUPS.items()}

# Mismo sanity check que en A.2: autovalores en [0, 2].
_x = torch.randn(len(ny_y), 8, device=DEVICE)
for name, lap in NY_LAPS.items():
    ratio = (lap.mm(_x).norm() / _x.norm()).item()
    assert ratio <= 2.0 + 1e-3, (name, ratio)
    print(f"  {name:<16} ||L x|| / ||x|| = {ratio:.3f}")
del _x


## B.3 Splits: aleatorio, agrupado (reviewer / negocio) y temporal

- **Aleatorio**: comparable con el 0,8448/0,3866 de LightGBM ya documentado.
- **Agrupado por `reviewer_id`**: ningún reviewer aparece en train y test a la vez.
- **Agrupado por `business_id`**: escenario de "negocio nunca visto" — el más
  exigente y el más parecido a un cliente nuevo.
- **Temporal**: train con lo más antiguo, test con lo más reciente. El más
  realista, aunque con la limitación honesta de que las features de
  reviewer/negocio se calculan sobre el dataset completo (transductivas) — no
  corrige esa fuga, solo corta la fuga de identidad de grupo y la de mirar el
  futuro para evaluar.

**Advertencia a asumir de antemano**: el 0,8448/0,3866 con split aleatorio ya
está inflado por esta razón. Es esperable, y sería un acierto de este notebook,
que las cifras aquí bajo protocolo estricto sean menores.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit


def split_random_np(y, test_frac, val_frac, seed=SEED):
    idx = np.arange(len(y))
    tr, rest = train_test_split(idx, train_size=1 - test_frac - val_frac, stratify=y, random_state=seed)
    va, te = train_test_split(rest, train_size=val_frac / (test_frac + val_frac),
                               stratify=y[rest], random_state=seed)
    return tr, va, te


def split_grouped_np(groups, test_frac, val_frac, seed=SEED):
    n = len(groups)
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    trainval_idx, te = next(gss1.split(np.zeros(n), groups=groups))
    rel_val = val_frac / (1 - test_frac)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=rel_val, random_state=seed)
    sub_groups = groups[trainval_idx]
    tr_rel, va_rel = next(gss2.split(np.zeros(len(trainval_idx)), groups=sub_groups))
    tr, va = trainval_idx[tr_rel], trainval_idx[va_rel]
    assert set(groups[tr]) & set(groups[te]) == set()
    assert set(groups[va]) & set(groups[te]) == set()
    assert set(groups[tr]) & set(groups[va]) == set()
    return tr, va, te


def split_temporal_np(date_days, test_frac, val_frac):
    order = np.argsort(date_days)
    unique_days = np.sort(np.unique(date_days))
    t1 = unique_days[int(len(unique_days) * (1 - test_frac - val_frac))]
    t2 = unique_days[int(len(unique_days) * (1 - test_frac))]
    tr = order[date_days[order] <= t1]
    va = order[(date_days[order] > t1) & (date_days[order] <= t2)]
    te = order[date_days[order] > t2]
    assert date_days[tr].max() <= date_days[va].min()
    assert date_days[va].max() <= date_days[te].min()
    return tr, va, te


NY_SPLITS = {
    "aleatorio": split_random_np(ny_y, test_frac=0.15, val_frac=0.15),
    "agrupado_reviewer": split_grouped_np(ny_reviewer_id, test_frac=0.15, val_frac=0.15),
    "agrupado_negocio": split_grouped_np(ny_business_id, test_frac=0.15, val_frac=0.15),
    "temporal": split_temporal_np(ny_date_days, test_frac=0.15, val_frac=0.15),
}
for name, (tr, va, te) in NY_SPLITS.items():
    print(f"{name:<18} train {len(tr):>7,}  val {len(va):>7,}  test {len(te):>7,}  "
          f"(base rate test {ny_y[te].mean():.4f})")

# Cold-start: reviews de reviewers con una unica review EN TODO EL DATASET.
# return_inverse (no indexar directamente por el valor crudo del id) para no
# asumir que ny_reviewer_id ya es un rango denso 0..N-1 -- es cierto para el
# bundle actual (pd.factorize lo garantiza) pero mejor no depender de ello.
_, _inv_rev, _counts_rev = np.unique(ny_reviewer_id, return_inverse=True, return_counts=True)
ny_cold_start_mask = (_counts_rev[_inv_rev] == 1)
print(f"\ncold-start (reviewers de 1 sola review): {ny_cold_start_mask.sum():,} "
      f"({ny_cold_start_mask.mean():.1%} del dataset)")


## B.4 Estandarización (ajustada solo en train, por split)

Las 24 features tienen escalas muy distintas (recuentos, ratios, entropías).
`StandardScaler` ajustado **solo con las filas de train** de cada split — nunca
con test ni val, para no filtrar su distribución.

In [ ]:
def standardize_by_train(X, train_idx):
    mean = X[train_idx].mean(axis=0)
    std = X[train_idx].std(axis=0)
    std[std < 1e-8] = 1.0
    return (X - mean) / std


## B.5 Entrenar y evaluar: MLP sin grafo vs. BWGNN hetero, en los 4 splits

Usa `BEST_CONFIG` de la Parte A (mismos `hidden`/`d`/`lr`/`weight_decay` que
ganaron en YelpChi) — es donde se cierra el círculo "mejorar en YelpChi, aplicar
en Yelp-NYC". `epochs=800, patience=100`: con solo 24 features de entrada y
propagación por scatter (mucho más barata que una sparse-mm sobre aristas
reales), cada época debería costar muy poco incluso con 359k nodos.

In [ ]:
def train_one_yelpnyc(X, y, laps_dict, model_name, model_cfg, split, cfg,
                        epochs=800, patience=100, seed=SEED, verbose=False):
    tr, va, te = split
    Xs = standardize_by_train(X, tr).astype(np.float32)

    torch.manual_seed(seed)
    np.random.seed(seed)

    laps = [laps_dict[r] for r in model_cfg["rels"]]
    if model_cfg["kind"] == "mlp":
        model = MLPNet(Xs.shape[1], cfg["hidden"], dropout=cfg.get("dropout", 0.1)).to(DEVICE)
    else:
        model = GraphNet(Xs.shape[1], cfg["hidden"], len(laps), bank_cls=BetaWaveletBank,
                          d=cfg["d"], dropout=cfg.get("dropout", 0.1)).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    n_pos = int(y[tr].sum())
    pos_weight = (len(tr) - n_pos) / max(1, n_pos)
    class_w = torch.tensor([1.0, pos_weight], dtype=torch.float32, device=DEVICE)

    X_t = torch.from_numpy(Xs).to(DEVICE)
    Y_t = torch.from_numpy(y).to(DEVICE)
    tr_t = torch.from_numpy(tr).to(DEVICE)

    best_ap, best_scores, best_epoch, no_improve = -1.0, None, -1, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        loss = F.cross_entropy(model(X_t, laps)[tr_t], Y_t[tr_t], weight=class_w)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            scores = F.softmax(model(X_t, laps), dim=1)[:, 1].float().cpu().numpy()
        ap_val = average_precision_score(y[va], scores[va])
        if ap_val > best_ap:
            best_ap, best_scores, best_epoch, no_improve = ap_val, scores.copy(), ep, 0
        else:
            no_improve += 1
        if verbose and (ep == 1 or ep % 100 == 0):
            print(f"    ep {ep:>4}  val_AP {ap_val:.4f}")
        if no_improve >= patience:
            break

    res = evaluate(y[te], best_scores[te])
    res["cold_start"] = None
    cold_te = ny_cold_start_mask[te]
    if cold_te.sum() >= 50 and y[te][cold_te].sum() >= 2:
        res["cold_start"] = evaluate(y[te][cold_te], best_scores[te][cold_te])
    res.update(model=model_name, best_epoch=best_epoch, epochs_ran=ep,
                val_average_precision=float(best_ap),
                train_seconds=round(time.time() - t0, 1), n_train=len(tr), n_test=len(te))
    return res


NY_MODELS = {
    "mlp_sin_grafo": MLP_CFG,
    "bwgnn_hetero": dict(kind="graph", rels=["rev", "biz_rating", "biz_rating_week"], bank=BetaWaveletBank),
}

NY_RESULTS = {}
t_all = time.time()
for split_name, split in NY_SPLITS.items():
    for model_name, model_cfg in NY_MODELS.items():
        key = f"{model_name}__{split_name}"
        print(f"=== {key}")
        NY_RESULTS[key] = train_one_yelpnyc(
            ny_X_raw, ny_y, NY_LAPS, model_name, model_cfg, split, BEST_CONFIG,
            epochs=800, patience=100, verbose=False,
        )
        r = NY_RESULTS[key]
        cs = r["cold_start"]
        cs_txt = f"  cold-start AUC {cs['roc_auc']:.4f} AP {cs['average_precision']:.4f}" if cs else "  cold-start: n/a"
        print(f"    AUC {r['roc_auc']:.4f}  AP {r['average_precision']:.4f}  "
              f"(mejor ep {r['best_epoch']}/{r['epochs_ran']}, {r['train_seconds']}s){cs_txt}")

print(f"\nTOTAL Parte B: {(time.time() - t_all) / 60:.1f} min")


## B.6 Comparación con el modelo de producción actual

Referencia: LightGBM sobre las mismas 24 features, split aleatorio simple —
**AUC 0,8448 / AP 0,3866**, cold-start **AUC 0,6632 / AP 0,3668**
(`outputs/metrics_behavior.json`, `features_behavior.py`).

In [ ]:
LGBM_REFERENCE = {
    "aleatorio": {"roc_auc": 0.8448, "average_precision": 0.3866},
    "cold_start": {"roc_auc": 0.6632, "average_precision": 0.3668},
}

rows = []
for key, r in NY_RESULTS.items():
    model_name, split_name = key.split("__")
    row = {
        "modelo": model_name, "split": split_name,
        "AUC": round(r["roc_auc"], 4), "AP": round(r["average_precision"], 4),
        "P@5%": round(r["top_5pct"]["precision"], 3), "lift@5%": round(r["top_5pct"]["lift"], 2),
        "epoca": r["best_epoch"], "seg": r["train_seconds"],
    }
    if r["cold_start"]:
        row["cold_AUC"] = round(r["cold_start"]["roc_auc"], 4)
        row["cold_AP"] = round(r["cold_start"]["average_precision"], 4)
    rows.append(row)

tabla_b = pd.DataFrame(rows).sort_values(["split", "AP"], ascending=[True, False])
print(f"Referencia LightGBM (split aleatorio simple): AUC {LGBM_REFERENCE['aleatorio']['roc_auc']} "
      f"AP {LGBM_REFERENCE['aleatorio']['average_precision']}")
print(f"Referencia LightGBM cold-start: AUC {LGBM_REFERENCE['cold_start']['roc_auc']} "
      f"AP {LGBM_REFERENCE['cold_start']['average_precision']}")
print()
tabla_b


## B.7 Guardar y descargar resultados

In [ ]:
import json

def _clean(obj):
    if isinstance(obj, dict):
        return {k: _clean(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_clean(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

payload = {
    "notebook": "colab/bwgnn_v2.ipynb",
    "seed": SEED,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_version": torch.__version__,
    "part_a_yelpchi": {
        "baseline_reproduction": baseline_res,
        "mejora_mas_epocas": mejora1_res,
        "grid_d_hidden": grid_a_results,
        "grid_lr_weight_decay": grid_b_results,
        "best_config": BEST_CONFIG,
        "seed_ensemble": {"n_seeds": N_SEEDS, "result": ensemble_res},
        "reference_baseline_v1": REF_BASELINE_40PCT,
    },
    "part_b_yelpnyc": {
        "n_nodes": int(len(ny_y)),
        "base_rate": float(ny_y.mean()),
        "n_cold_start": int(ny_cold_start_mask.sum()),
        "relations": list(NY_GROUPS.keys()),
        "best_config_reused_from_part_a": BEST_CONFIG,
        "results": NY_RESULTS,
        "lgbm_reference": LGBM_REFERENCE,
    },
    "protocol": (
        "Splits estratificados (aleatorio) o agrupados (GroupShuffleSplit por "
        "reviewer_id/business_id) o temporales (por fecha). Epoca elegida por AP "
        "en validacion, nunca en test. Sin target encoding: ninguna feature usa "
        "is_fake para calcularse. Las features de reviewer/negocio son "
        "transductivas (se calculan sobre el dataset completo), lo que infla algo "
        "incluso el split temporal -- limitacion conocida, no corregida en este "
        "notebook."
    ),
}
payload = _clean(payload)

OUT_PATH = Path("bwgnn_v2_results.json")
OUT_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Guardado en {OUT_PATH.resolve()}  ({OUT_PATH.stat().st_size / 1024:.1f} KB)")


In [ ]:
try:
    from google.colab import files
    files.download(str(OUT_PATH))
    print("Descarga lanzada. Si el navegador la bloquea, el fichero esta en el panel")
    print("de la izquierda (icono de carpeta) como bwgnn_v2_results.json.")
except ImportError:
    print("No estamos en Colab; el JSON queda en", OUT_PATH.resolve())


---
## Siguientes pasos (no en este notebook)

- **Fase 2 del plan**: agregación de vecindario sobre árboles (XGB-Graph, sin GNN) —
  GADBench mide que esta vía bate a BWGNN en +12,9 puntos de AUPRC de media, así
  que sigue siendo la comparación pendiente más importante.
- **Fase 3**: arreglar la señal de texto (hoy casi no aporta:
  `text_max_sim_business_diff_reviewer` tiene AUC 0,4741, por debajo del azar).
- **Fase 4**: ensemble final LightGBM + XGBoost + CatBoost + BWGNN.

Trae de vuelta `bwgnn_v2_results.json` a la conversación con Claude Code para
incorporar estos números a `CONTEXTO.md` y decidir el siguiente paso.